# Image Processing - Vision Transformers on Colab

**Secure image classification and processing with 100% privacy**

![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)

---

## Supported Tasks

- \ud83d\udd0d Image Classification
- \ud83d\udca1 Object Detection
- \ud83d\udcf8 Image Segmentation
- \ud83d\udc40 Visual Question Answering (VQA)

---


In [ ]:
# Model configuration
MODEL_TYPE = "classification"  # @param ["classification", "detection", "segmentation", "vqa"]
MODEL_ID = "microsoft/resnet-50"  # @param {type:"string"}

print(f"\ud83d\udc49 Model: {MODEL_ID}")
print(f"\ud83d\udc49 Task: {MODEL_TYPE}")

In [ ]:
# Security setup
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = "/content/.hf_cache"
!mkdir -p /content/.hf_cache

# Install
!pip install -q transformers accelerate torchvision pillow requests

print("\u2705 Setup complete!")

In [ ]:
# Load vision model
from transformers import AutoImageProcessor, AutoModelForImageClassification

print(f"Loading {MODEL_ID}...")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

print(f"\u2705 Model loaded: {len(processor.id2label)} classes")

In [ ]:
from PIL import Image
import requests
import torch

def classify_image(image_path):
    """Classify image with secure processing."""
    
    # Load image
    if image_path.startswith('http'):
        image = Image.open(requests.get(image_path, stream=True).raw)
    else:
        image = Image.open(image_path)
    
    # Process
    inputs = processor(images=image, return_tensors="pt")
    
    # Classify
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get top prediction
    probs = torch.nn.functional.softmax(outputs.logits[0], dim=-1)
    top_prob, top_idx = probs.topk(1)
    
    label = processor.id2label[top_idx.item()]
    confidence = top_prob.item() * 100
    
    return label, confidence

print("\u2705 Classification ready!")

In [ ]:
# Test with sample image
test_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/be/Border_Collie_-_Aristotle.jpg/1200px-Border_Collie_-_Aristotle.jpg"

print(f"Testing: {test_url}\n")
label, confidence = classify_image(test_url)
print(f"\ud83d\udc51 Result: {label}")
print(f"\ud83d\udcc8 Confidence: {confidence:.1f}%")

In [ ]:
# Cleanup
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\u2705 Memory cleared!")

---

**Popular Vision Models**:
- ResNet-50, EfficientNet, ViT (classification)
- DETR, YOLOS (detection)
- CLIP (zero-shot classification)

**Star**: [GitHub](https://github.com/unn-Known1/huggingface-colab-secure-setup)